In [1]:
!pip install blosum

In [2]:
import pandas as pd
import numpy as np
import torch
import re
import lightgbm as lgb
from tqdm import tqdm
from transformers import AutoTokenizer, EsmForMaskedLM
from sklearn.model_selection import KFold
from scipy.stats import spearmanr

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
!head /kaggle/input/datasets/luciferzmh/ml-4-bio/query1_result.csv

mutant,DMS_score,sequence
N11Y,0.9581050797902574,MVNEARGNSSLYPCLEGSASSGSESSKDSSRCSTPGLDPERHERLREKMRRRLESGDKWFSLEFFPPRTAEGAVNLISRFDRMAAGGPLYIDVTWHPAGDPGSDKETSSMMIASTAVNYCGLETILHMTCCRQRLEEITGHLHKAKQLGLKNIMALRGDPIGDQWEEEEGGFNYAVDLVKHIRSEFGDYFDICVAGYPKGHPEAGSFEADLKHLKEKVSAGADFIITQLFFEADTFFRFVKACTDMGITCPIVPGIFPIQGYHSLRQLVKLSKLEVPQEIKDVIEPIKDNDAAIRNYGIELAVSLCQELLASGLVPGLHFYTLNREMATTEVLKRLGMWTEDPRRPLPWALSAHPKRREEDVRPIFWASRPKSYIYRTQEWDEFPNGRWGNSSSPAFGELKDYYLFYLKSKSPKEELLKMWGEELTSEESVFEVFVLYLSGEPNRNGHKVTCLPWNDEPLAAETSLLKEELLRVNRQGILTINSQPNINGKPSSDPIVGWGPSGGYVFQKAYLEFFTSRETAEALLQVLKKYELRVNYHLVNVKGENITNAPELQPNAVTWGIFPGREIIQPTVVDPVSFMFWKDEAFALWIERWGKLYEEESPSRTIIQYIHDNYFLVNLVDNDFPLDNCLWQVVEDTLELLNRPTQNARETEAP
N11C,0.6976208147822618,MVNEARGNSSLCPCLEGSASSGSESSKDSSRCSTPGLDPERHERLREKMRRRLESGDKWFSLEFFPPRTAEGAVNLISRFDRMAAGGPLYIDVTWHPAGDPGSDKETSSMMIASTAVNYCGLETILHMTCCRQRLEEITGHLHKAKQLGLKNIMALRGDPIGDQWEEEEGGFNYAVDLVKHIRSEFGDYFDICVAGYPKGHPEAGSFEADLKHLKEKVSAGADFIITQLFFEADTFFRFVKACTDMGITCPIVPGIFPIQGYHSLRQLV

In [4]:
fasta_path = '/kaggle/input/datasets/luciferzmh/ml-4-bio/sequence.fasta'
with open(fasta_path, 'r') as f:
    lines = f.readlines()
    wt_seq = "".join([line.strip() for line in lines if not line.startswith(">")])

# merge
train_df = pd.read_csv('/kaggle/input/datasets/luciferzmh/ml-4-bio/train.csv')
query1_df = pd.read_csv('/kaggle/input/datasets/luciferzmh/ml-4-bio/query1_result.csv')
train_df = pd.concat([train_df, query1_df[['mutant', 'DMS_score']]], ignore_index=True)
query2_df = pd.read_csv('/kaggle/input/datasets/luciferzmh/ml-4-bio/query2_result.csv')
train_df = pd.concat([train_df, query2_df[['mutant', 'DMS_score']]], ignore_index=True)
query3_df = pd.read_csv('/kaggle/input/datasets/luciferzmh/ml-4-bio/query3_result.csv')
train_df = pd.concat([train_df, query3_df[['mutant', 'DMS_score']]], ignore_index=True)

test_df = pd.read_csv('/kaggle/input/datasets/luciferzmh/ml-4-bio/test.csv')

def parse_mutant(mutant_str):
    "M(WT) 0(Pos) Y(Mut)"""
    wt_aa = mutant_str[0]
    pos = int(re.findall(r'\d+', mutant_str)[0])
    mut_aa = mutant_str[-1]
    return wt_aa, pos, mut_aa

print(f"Wild-type sequence length: {len(wt_seq)}")

Wild-type sequence length: 656


In [5]:
model_name = "facebook/esm2_t33_650M_UR50D" 
tokenizer = AutoTokenizer.from_pretrained(model_name)
#ForMaskedLM
model = EsmForMaskedLM.from_pretrained(model_name).to(device)
model.eval()

config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.61G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/571 [00:00<?, ?it/s]

EsmForMaskedLM LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                         | Status     |  | 
----------------------------+------------+--+-
esm.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


EsmForMaskedLM(
  (esm): EsmModel(
    (embeddings): EsmEmbeddings(
      (word_embeddings): Embedding(33, 1280, padding_idx=1)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): EsmEncoder(
      (layer): ModuleList(
        (0-32): 33 x EsmLayer(
          (attention): EsmAttention(
            (self): EsmSelfAttention(
              (query): Linear(in_features=1280, out_features=1280, bias=True)
              (key): Linear(in_features=1280, out_features=1280, bias=True)
              (value): Linear(in_features=1280, out_features=1280, bias=True)
              (rotary_embeddings): RotaryEmbedding()
            )
            (output): EsmSelfOutput(
              (dense): Linear(in_features=1280, out_features=1280, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (LayerNorm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          )
          (intermediate): EsmIntermediate(
            (dense): Linear(in_features=1

In [6]:
import numpy as np
import torch
from tqdm import tqdm

def precompute_esm2_matrices(wt_sequence):
    # 1. get original Embedding
    wt_inputs = tokenizer(wt_sequence, return_tensors="pt").to(device)
    with torch.no_grad():
        wt_outputs = model(**wt_inputs, output_hidden_states=True)
        wt_embeddings = wt_outputs.hidden_states[-2][0].cpu().numpy() 

    # 2. calculate Masked Marginals
    seq_len = len(wt_sequence)
    vocab_size = len(tokenizer)
    masked_logits_matrix = np.zeros((seq_len, vocab_size))
    
    print("Computing True Masked Marginals...")
    for pos in tqdm(range(seq_len)):
        masked_seq = wt_sequence[:pos] + tokenizer.mask_token + wt_sequence[pos+1:]
        inputs = tokenizer(masked_seq, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            masked_logits_matrix[pos, :] = outputs.logits[0, pos + 1].cpu().numpy()
            
    return wt_embeddings, masked_logits_matrix

vocab_dict = {aa: tokenizer.convert_tokens_to_ids(aa) for aa in "ACDEFGHIKLMNPQRSTVWY"}

def build_features_from_matrices(mutant_list, wt_embeddings, masked_logits_matrix):
    n_samples = len(mutant_list)
    emb_dim = wt_embeddings.shape[1]
    
    features_matrix = np.zeros((n_samples, 1 + emb_dim), dtype=np.float32)
    
    for i, mutant in enumerate(tqdm(mutant_list, desc="Extracting Features")):
        wt_aa, pos, mut_aa = parse_mutant(mutant)
        
        wt_id = vocab_dict.get(wt_aa, tokenizer.convert_tokens_to_ids(wt_aa))
        mut_id = vocab_dict.get(mut_aa, tokenizer.convert_tokens_to_ids(mut_aa))
        
        true_llr = masked_logits_matrix[pos, mut_id] - masked_logits_matrix[pos, wt_id]
        wt_emb = wt_embeddings[pos + 1] # <cls> pos + 1
        
        features_matrix[i, 0] = true_llr
        features_matrix[i, 1:] = wt_emb
        
    return features_matrix

# cache matrix
wt_embeddings, masked_logits_matrix = precompute_esm2_matrices(wt_seq)

print("Processing Training and Test Data...")
X_train_all = build_features_from_matrices(train_df['mutant'].tolist(), wt_embeddings, masked_logits_matrix)
y_train_all = train_df['DMS_score'].values

X_test_all = build_features_from_matrices(test_df['mutant'].tolist(), wt_embeddings, masked_logits_matrix)

Computing True Masked Marginals...


100%|██████████| 656/656 [02:02<00:00,  5.35it/s]


Processing Training and Test Data...


Extracting Features: 100%|██████████| 11324/11324 [00:00<00:00, 82292.84it/s]


In [7]:
import blosum as bl
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.stats import spearmanr
import lightgbm as lgb
from sklearn.model_selection import KFold

# 1. 准备手工特征字典 (BLOSUM, 亲疏水性, 分子量)
matrix = bl.BLOSUM(62)

kd_scale = {
    'A': 1.8, 'R': -4.5, 'N': -3.5, 'D': -3.5, 'C': 2.5, 'Q': -3.5, 'E': -3.5, 'G': -0.4, 
    'H': -3.2, 'I': 4.5, 'L': 3.8, 'K': -3.9, 'M': 1.9, 'F': 2.8, 'P': -1.6, 'S': -0.8, 
    'T': -0.7, 'W': -0.9, 'Y': -1.3, 'V': 4.2
}

aa_weights = {
    'A': 89.09, 'R': 174.20, 'N': 132.12, 'D': 133.10, 'C': 121.16, 'Q': 146.15, 'E': 147.13, 
    'G': 75.07, 'H': 155.16, 'I': 131.18, 'L': 131.18, 'K': 146.19, 'M': 149.21, 'F': 165.19, 
    'P': 115.13, 'S': 105.09, 'T': 119.12, 'W': 204.23, 'Y': 181.19, 'V': 117.15
}

# 2. 极速提取综合手工特征的函数
def build_handcrafted_features(mutant_list, seq_len):
    features = []
    for mut in mutant_list:
        wt_aa, mut_aa = mut[0], mut[-1]
        pos = int(mut[1:-1])

        # Feature 1: BLOSUM62
        try:
            blosum_score = matrix[wt_aa][mut_aa]
        except:
            blosum_score = 0
            
        # Feature 2 & 3: PhysChem Deltas
        mw_delta = aa_weights.get(mut_aa, 0) - aa_weights.get(wt_aa, 0)
        hydro_delta = kd_scale.get(mut_aa, 0) - kd_scale.get(wt_aa, 0)
        
        # Feature 4 & 5: Positional
        rel_pos = pos / seq_len
        wt_idx = tokenizer.convert_tokens_to_ids(wt_aa) # 保持与 ESM 一致的 ID
        
        features.append([blosum_score, mw_delta, hydro_delta, rel_pos, wt_idx])
    return np.array(features)

print("提取手工特征 (Handcrafted Features)...")
HC_train = build_handcrafted_features(train_df['mutant'].tolist(), len(wt_seq))
HC_test = build_handcrafted_features(test_df['mutant'].tolist(), len(wt_seq))

# 3. 基础特征拆分与 PCA
LLR_train = X_train_all[:, 0].reshape(-1, 1)
Emb_train = X_train_all[:, 1:]
LLR_test = X_test_all[:, 0].reshape(-1, 1)
Emb_test = X_test_all[:, 1:]

print("Applying PCA to embeddings...")
pca = PCA(n_components=50, random_state=42)
Emb_train_pca = pca.fit_transform(Emb_train)
Emb_test_pca = pca.transform(Emb_test)

# 4. 终极特征拼接: LLR (1) + PCA (50) + Handcrafted (5) = 56 维
X_train_compact = np.hstack([LLR_train, Emb_train_pca, HC_train])
X_test_compact = np.hstack([LLR_test, Emb_test_pca, HC_test])

print(f"✅ 拼接完成! 最终特征维度: {X_train_compact.shape[1]}")

# 5. 标准化与模型训练
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_compact)
X_test_scaled = scaler.transform(X_test_compact)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
final_test_preds = np.zeros(len(test_df))
oof_preds = np.zeros(len(train_df))

models_ridge = []
models_lgb = []

for fold, (t_idx, v_idx) in enumerate(kf.split(X_train_scaled)):
    X_t, X_v = X_train_scaled[t_idx], X_train_scaled[v_idx]
    y_t, y_v = y_train_all[t_idx], y_train_all[v_idx]
    
    ridge = RidgeCV(alphas=[0.1, 1.0, 10.0, 50.0, 100.0])
    ridge.fit(X_t, y_t)
    
    # 注意: 为了防止死锁，这里的 device='cpu'
    lgb_reg = lgb.LGBMRegressor(
        n_estimators=1000, learning_rate=0.01, num_leaves=15,
        feature_fraction=0.8, lambda_l1=1.0, lambda_l2=1.0,
        device='cpu', verbosity=-1
    )
    lgb_reg.fit(X_t, y_t, eval_set=[(X_v, y_v)], callbacks=[lgb.early_stopping(50)])
    
    models_ridge.append(ridge)
    models_lgb.append(lgb_reg)
    
    pred_ridge = ridge.predict(X_v)
    pred_lgb = lgb_reg.predict(X_v)
    combined_pred = 0.6 * pred_ridge + 0.4 * pred_lgb
    
    oof_preds[v_idx] = combined_pred
    final_test_preds += (0.6 * ridge.predict(X_test_scaled) + 0.4 * lgb_reg.predict(X_test_scaled)) / 5
    
    print(f"Fold {fold+1} Spearman: {spearmanr(y_v, combined_pred)[0]:.4f}")

print(f"\nOverall CV Spearman Correlation: {spearmanr(y_train_all, oof_preds)[0]:.4f}")

提取手工特征 (Handcrafted Features)...
Applying PCA to embeddings...
✅ 拼接完成! 最终特征维度: 56
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[776]	valid_0's l2: 0.0294848


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 1 Spearman: 0.7014
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[987]	valid_0's l2: 0.0255596


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 2 Spearman: 0.7266
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[423]	valid_0's l2: 0.02835
Fold 3 Spearman: 0.6951
Training until validation scores don't improve for 50 rounds


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Early stopping, best iteration is:
[542]	valid_0's l2: 0.0263989
Fold 4 Spearman: 0.7157


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[441]	valid_0's l2: 0.0304283
Fold 5 Spearman: 0.7814

Overall CV Spearman Correlation: 0.7274


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [8]:
'''
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.stats import spearmanr
import lightgbm as lgb
from sklearn.model_selection import KFold

LLR_train = X_train_all[:, 0].reshape(-1, 1)
Emb_train = X_train_all[:, 1:]
LLR_test = X_test_all[:, 0].reshape(-1, 1)
Emb_test = X_test_all[:, 1:]

print("Applying PCA to reduce embedding dimensions...")
pca = PCA(n_components=50, random_state=42)
Emb_train_pca = pca.fit_transform(Emb_train)
Emb_test_pca = pca.transform(Emb_test)

X_train_compact = np.hstack([LLR_train, Emb_train_pca])
X_test_compact = np.hstack([LLR_test, Emb_test_pca])

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_compact)
X_test_scaled = scaler.transform(X_test_compact)

kf = KFold(n_splits=20, shuffle=True, random_state=2345)
final_test_preds = np.zeros(len(test_df))
oof_preds = np.zeros(len(train_df))

# form a committee using every fold
models_ridge = []
models_lgb = []

for fold, (t_idx, v_idx) in enumerate(kf.split(X_train_scaled)):
    X_t, X_v = X_train_scaled[t_idx], X_train_scaled[v_idx]
    y_t, y_v = y_train_all[t_idx], y_train_all[v_idx]
    
    ridge = RidgeCV(alphas=[0.1, 1.0, 10.0, 50.0, 100.0])
    ridge.fit(X_t, y_t)
    
    lgb_reg = lgb.LGBMRegressor(
        n_estimators=1000, learning_rate=0.01, num_leaves=15,
        feature_fraction=0.8, lambda_l1=1.0, lambda_l2=1.0,
        device='gpu', verbosity=-1
    )
    lgb_reg.fit(X_t, y_t, eval_set=[(X_v, y_v)], callbacks=[lgb.early_stopping(50)])
    
    models_ridge.append(ridge)
    models_lgb.append(lgb_reg)
    
    pred_ridge = ridge.predict(X_v)
    pred_lgb = lgb_reg.predict(X_v)
    combined_pred = 0.8 * pred_ridge + 0.2 * pred_lgb
    
    oof_preds[v_idx] = combined_pred
    final_test_preds += (0.8 * ridge.predict(X_test_scaled) + 0.2 * lgb_reg.predict(X_test_scaled)) / 5
    
    print(f"Fold {fold+1} Spearman: {spearmanr(y_v, combined_pred)[0]:.4f}")

print(f"\nOverall CV Spearman Correlation: {spearmanr(y_train_all, oof_preds)[0]:.4f}")
'''

'\nfrom sklearn.linear_model import RidgeCV\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.decomposition import PCA\nfrom scipy.stats import spearmanr\nimport lightgbm as lgb\nfrom sklearn.model_selection import KFold\n\nLLR_train = X_train_all[:, 0].reshape(-1, 1)\nEmb_train = X_train_all[:, 1:]\nLLR_test = X_test_all[:, 0].reshape(-1, 1)\nEmb_test = X_test_all[:, 1:]\n\nprint("Applying PCA to reduce embedding dimensions...")\npca = PCA(n_components=50, random_state=42)\nEmb_train_pca = pca.fit_transform(Emb_train)\nEmb_test_pca = pca.transform(Emb_test)\n\nX_train_compact = np.hstack([LLR_train, Emb_train_pca])\nX_test_compact = np.hstack([LLR_test, Emb_test_pca])\n\nscaler = StandardScaler()\nX_train_scaled = scaler.fit_transform(X_train_compact)\nX_test_scaled = scaler.transform(X_test_compact)\n\nkf = KFold(n_splits=20, shuffle=True, random_state=2345)\nfinal_test_preds = np.zeros(len(test_df))\noof_preds = np.zeros(len(train_df))\n\n# form a committee using every

In [9]:
'''
#query 1


import numpy as np

# 1. generate whole aa space
amino_acids = list("ACDEFGHIKLMNPQRSTVWY")
all_single_mutants = []
for pos, wt_aa in enumerate(wt_seq):
    for mut_aa in amino_acids:
        if wt_aa != mut_aa:
            all_single_mutants.append(f"{wt_aa}{pos}{mut_aa}")

# 2. filter train data
known_mutants = set(train_df['mutant'])
candidate_mutants = [m for m in all_single_mutants if m not in known_mutants]

print(f"All muts: {len(all_single_mutants)}")
print(f"Muts (Train): {len(known_mutants)}")
print(f"Muts for query: {len(candidate_mutants)}")

# 3. using cached ESM-2 matrix
print("Building features for candidates...")
X_cand_raw = build_features_from_matrices(candidate_mutants, wt_embeddings, masked_logits_matrix)

# 4. alighment
LLR_cand = X_cand_raw[:, 0].reshape(-1, 1)
Emb_cand = X_cand_raw[:, 1:]

# PCA
Emb_cand_pca = pca.transform(Emb_cand)
X_cand_compact = np.hstack([LLR_cand, Emb_cand_pca])

# Scaler
X_cand_scaled = scaler.transform(X_cand_compact)

# 5. Query by Committee (QBC)
print("Predicting with ensemble to calculate uncertainty...")
preds_matrix = []

# Iterate through all models saved from 5-fold cross-validation
for ridge, lgb_reg in zip(models_ridge, models_lgb):
    pred_r = ridge.predict(X_cand_scaled)
    pred_l = lgb_reg.predict(X_cand_scaled)
    # Maintain mixed weights consistent with the validation set
    preds_matrix.append(0.6 * pred_r + 0.4 * pred_l)

preds_matrix = np.array(preds_matrix) # Dimension: (5, num_candidates)
# Calculate the variance of each candidate mutation across the predictions of the 5 models
uncertainty = np.var(preds_matrix, axis=0) 

# 6. Get top 100
top_100_indices = np.argsort(uncertainty)[-100:][::-1]
top_100_mutants = [candidate_mutants[i] for i in top_100_indices]

# 7. query.txt
with open('query.txt', 'w') as f:
    for m in top_100_mutants:
        f.write(m + '\n')

print("✅ Active learning complete! Top 100 uncertain mutations saved to 'query.txt'.")

'''

'\n#query 1\n\n\nimport numpy as np\n\n# 1. generate whole aa space\namino_acids = list("ACDEFGHIKLMNPQRSTVWY")\nall_single_mutants = []\nfor pos, wt_aa in enumerate(wt_seq):\n    for mut_aa in amino_acids:\n        if wt_aa != mut_aa:\n            all_single_mutants.append(f"{wt_aa}{pos}{mut_aa}")\n\n# 2. filter train data\nknown_mutants = set(train_df[\'mutant\'])\ncandidate_mutants = [m for m in all_single_mutants if m not in known_mutants]\n\nprint(f"All muts: {len(all_single_mutants)}")\nprint(f"Muts (Train): {len(known_mutants)}")\nprint(f"Muts for query: {len(candidate_mutants)}")\n\n# 3. using cached ESM-2 matrix\nprint("Building features for candidates...")\nX_cand_raw = build_features_from_matrices(candidate_mutants, wt_embeddings, masked_logits_matrix)\n\n# 4. alighment\nLLR_cand = X_cand_raw[:, 0].reshape(-1, 1)\nEmb_cand = X_cand_raw[:, 1:]\n\n# PCA\nEmb_cand_pca = pca.transform(Emb_cand)\nX_cand_compact = np.hstack([LLR_cand, Emb_cand_pca])\n\n# Scaler\nX_cand_scaled = sc

In [10]:
'''
import numpy as np

#UCB parameter
kappa = 2.0 

# 1. generate whole aa space
amino_acids = list("ACDEFGHIKLMNPQRSTVWY")
all_single_mutants = []
for pos, wt_aa in enumerate(wt_seq):
    for mut_aa in amino_acids:
        if wt_aa != mut_aa:
            all_single_mutants.append(f"{wt_aa}{pos}{mut_aa}")

known_mutants = set(train_df['mutant'])
candidate_mutants = [m for m in all_single_mutants if m not in known_mutants]

print(f"Theoratical muts: {len(all_single_mutants)}")
print(f"Labeled muts (Train + Query 1): {len(known_mutants)}")
print(f"Muts for query: {len(candidate_mutants)}")

# 3. using cached ESM-2 matrix
print("Building features for candidates (Fast Mode)...")
X_cand_raw = build_features_from_matrices(candidate_mutants, wt_embeddings, masked_logits_matrix)

# 4. alignment
LLR_cand = X_cand_raw[:, 0].reshape(-1, 1)
Emb_cand = X_cand_raw[:, 1:]

# PCA
Emb_cand_pca = pca.transform(Emb_cand)
X_cand_compact = np.hstack([LLR_cand, Emb_cand_pca])

# Scaler
X_cand_scaled = scaler.transform(X_cand_compact)

# 5. Upper Confidence Bound (UCB) Evaluation
print(f"Predicting with ensemble to calculate UCB scores (kappa={kappa})...")
preds_matrix = []

# Iterate through all models saved from 5-fold cross-validation
for ridge, lgb_reg in zip(models_ridge, models_lgb):
    pred_r = ridge.predict(X_cand_scaled)
    pred_l = lgb_reg.predict(X_cand_scaled)
    # Maintain mixed weights consistent with the validation set
    preds_matrix.append(0.6 * pred_r + 0.4 * pred_l)

preds_matrix = np.array(preds_matrix) # Dimension: (5, num_candidates)

mean_preds = np.mean(preds_matrix, axis=0)
std_preds = np.std(preds_matrix, axis=0)

ucb_scores = mean_preds + kappa * std_preds

# 6. Get top 100 by UCB Score
top_100_indices = np.argsort(ucb_scores)[-100:][::-1]
top_100_mutants = [candidate_mutants[i] for i in top_100_indices]

# 7. query2.txt
with open('query2.txt', 'w') as f:
    for m in top_100_mutants:
        f.write(m + '\n')

print("✅ Active learning Round 2 complete! Top 100 UCB mutations saved to 'query2.txt'.")
'''

'\nimport numpy as np\n\n#UCB parameter\nkappa = 2.0 \n\n# 1. generate whole aa space\namino_acids = list("ACDEFGHIKLMNPQRSTVWY")\nall_single_mutants = []\nfor pos, wt_aa in enumerate(wt_seq):\n    for mut_aa in amino_acids:\n        if wt_aa != mut_aa:\n            all_single_mutants.append(f"{wt_aa}{pos}{mut_aa}")\n\nknown_mutants = set(train_df[\'mutant\'])\ncandidate_mutants = [m for m in all_single_mutants if m not in known_mutants]\n\nprint(f"Theoratical muts: {len(all_single_mutants)}")\nprint(f"Labeled muts (Train + Query 1): {len(known_mutants)}")\nprint(f"Muts for query: {len(candidate_mutants)}")\n\n# 3. using cached ESM-2 matrix\nprint("Building features for candidates (Fast Mode)...")\nX_cand_raw = build_features_from_matrices(candidate_mutants, wt_embeddings, masked_logits_matrix)\n\n# 4. alignment\nLLR_cand = X_cand_raw[:, 0].reshape(-1, 1)\nEmb_cand = X_cand_raw[:, 1:]\n\n# PCA\nEmb_cand_pca = pca.transform(Emb_cand)\nX_cand_compact = np.hstack([LLR_cand, Emb_cand_pca])

In [11]:
'''
import numpy as np
import pandas as pd


gamma = 0.35
max_per_position = 2 


amino_acids = list("ACDEFGHIKLMNPQRSTVWY")
all_single_mutants = []
for pos, wt_aa in enumerate(wt_seq):
    for mut_aa in amino_acids:
        if wt_aa != mut_aa:
            all_single_mutants.append(f"{wt_aa}{pos}{mut_aa}")

known_mutants = set(train_df['mutant'])
candidate_mutants = [m for m in all_single_mutants if m not in known_mutants]


print("Building features for candidates (Fast Mode)...")
X_cand_raw = build_features_from_matrices(candidate_mutants, wt_embeddings, masked_logits_matrix)

LLR_cand = X_cand_raw[:, 0].reshape(-1, 1)
Emb_cand = X_cand_raw[:, 1:]
Emb_cand_pca = pca.transform(Emb_cand)
X_cand_compact = np.hstack([LLR_cand, Emb_cand_pca])
X_cand_scaled = scaler.transform(X_cand_compact)

print("Predicting with existing ensemble...")
preds_matrix = []
for ridge, lgb_reg in zip(models_ridge, models_lgb):
    pred_r = ridge.predict(X_cand_scaled)
    pred_l = lgb_reg.predict(X_cand_scaled)
    preds_matrix.append(0.6 * pred_r + 0.4 * pred_l)

preds_matrix = np.array(preds_matrix)
mean_preds = np.mean(preds_matrix, axis=0)

query3_df = pd.DataFrame({
    "mutant": candidate_mutants,
    "predicted_fitness": mean_preds
})

query3_df["position"] = [int(m[1:-1]) for m in candidate_mutants]

position_importance = (
    query3_df.groupby("position")["predicted_fitness"]
    .var()
    .fillna(0.0)
    .rename("position_importance")
    .reset_index()
)

query3_df = query3_df.merge(position_importance, on="position", how="left")

#Selection Score
fitness_min = query3_df["predicted_fitness"].min()
fitness_max = query3_df["predicted_fitness"].max()
importance_min = query3_df["position_importance"].min()
importance_max = query3_df["position_importance"].max()

query3_df["fitness_norm"] = (query3_df["predicted_fitness"] - fitness_min) / (fitness_max - fitness_min + 1e-8)
query3_df["importance_norm"] = (query3_df["position_importance"] - importance_min) / (importance_max - importance_min + 1e-8)

query3_df["selection_score"] = query3_df["fitness_norm"] + gamma * query3_df["importance_norm"]

query3_df = query3_df.sort_values(["selection_score", "predicted_fitness"], ascending=False)

selected_rows = []
position_counts = {}

for _, row in query3_df.iterrows():
    pos = int(row["position"])
    count = position_counts.get(pos, 0)
    
    if count >= max_per_position:
        continue

    selected_rows.append(row)
    position_counts[pos] = count + 1

    if len(selected_rows) >= 100:
        break

top_100_mutants = pd.DataFrame(selected_rows).copy()
top_100_mutants.to_csv("query3_candidates_top100.csv", index=False)

with open("query3.txt", "w") as f:
    for mutant in top_100_mutants["mutant"].values:
        f.write(mutant + "\n")

print("✅ Query 3 (Positional Diversity + Exploitation) complete!")
print("Saved ranked top-100 table to query3_candidates_top100.csv")
print("Saved query list to query3.txt")

# display(query3_top100[["mutant", "position", "predicted_fitness", "position_importance", "selection_score"]].head(10))
'''

'\nimport numpy as np\nimport pandas as pd\n\n\ngamma = 0.35\nmax_per_position = 2 \n\n\namino_acids = list("ACDEFGHIKLMNPQRSTVWY")\nall_single_mutants = []\nfor pos, wt_aa in enumerate(wt_seq):\n    for mut_aa in amino_acids:\n        if wt_aa != mut_aa:\n            all_single_mutants.append(f"{wt_aa}{pos}{mut_aa}")\n\nknown_mutants = set(train_df[\'mutant\'])\ncandidate_mutants = [m for m in all_single_mutants if m not in known_mutants]\n\n\nprint("Building features for candidates (Fast Mode)...")\nX_cand_raw = build_features_from_matrices(candidate_mutants, wt_embeddings, masked_logits_matrix)\n\nLLR_cand = X_cand_raw[:, 0].reshape(-1, 1)\nEmb_cand = X_cand_raw[:, 1:]\nEmb_cand_pca = pca.transform(Emb_cand)\nX_cand_compact = np.hstack([LLR_cand, Emb_cand_pca])\nX_cand_scaled = scaler.transform(X_cand_compact)\n\nprint("Predicting with existing ensemble...")\npreds_matrix = []\nfor ridge, lgb_reg in zip(models_ridge, models_lgb):\n    pred_r = ridge.predict(X_cand_scaled)\n    pred_

In [12]:
submission = pd.DataFrame({
    'id': test_df['id'],
    'DMS_score': final_test_preds
})
submission.to_csv('submission.csv', index=False)
print("Submission file saved successfully!")

Submission file saved successfully!


In [13]:
!ls
!head submission.csv

__notebook__.ipynb  submission.csv
id,DMS_score
0,0.6029716467791091
1,0.6214836029070486
2,0.6224831056676141
3,0.6653095783092919
4,0.6277889042230175
5,0.5774113748295362
6,0.6568434139974662
7,0.5966100118428704
8,0.6138182405126048
